In [1]:
# Validação da Limpeza — VoughtGuard
#
# Este notebook valida manualmente, passo a passo, o que cada função de
# `src/limpeza.py` faz sobre uma amostra dos dados brutos.

In [2]:
%matplotlib inline

import sys
sys.path.append("..")

import pandas as pd
from src import limpeza

pd.set_option("display.max_columns", None)

In [3]:
df_bruto = limpeza.carregar_dados("../data/raw/transactions.csv")
print(f"Shape bruto: {df_bruto.shape}")
df_bruto.head()

Shape bruto: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
# ## 1. Validação de schema


In [5]:
faltantes = limpeza.validar_schema(df_bruto)
print(f"Colunas faltantes: {faltantes}")
assert faltantes == [], "Existem colunas obrigatórias faltando!"

Colunas faltantes: []


In [6]:
# ## 2. Duplicatas


In [7]:
total_antes = len(df_bruto)
duplicatas = df_bruto.duplicated().sum()
print(f"Total de linhas: {total_antes}")
print(f"Linhas duplicadas encontradas: {duplicatas}")

df_sem_duplicatas = limpeza.remover_duplicatas(df_bruto)
print(f"Total após remoção: {len(df_sem_duplicatas)}")

Total de linhas: 6362620
Linhas duplicadas encontradas: 0
Total após remoção: 6362620


In [8]:
# ## 3. Valores nulos

In [9]:
nulos_antes = df_sem_duplicatas.isnull().sum()
print("Nulos por coluna:")
print(nulos_antes[nulos_antes > 0])
df_sem_nulos = limpeza.tratar_nulos(df_sem_duplicatas)
print(f"\nTotal após tratamento de nulos: {len(df_sem_nulos)}")

Nulos por coluna:
Series([], dtype: int64)

Total após tratamento de nulos: 6362620


In [10]:
# ## 4. Valores inválidos (amount <= 0)


In [11]:
invalidos = df_sem_nulos[df_sem_nulos["amount"] <= 0]
print(f"Transações com amount <= 0: {len(invalidos)}")
print(f"Dessas, fraudes: {invalidos['isFraud'].sum()}")
invalidos.head()

Transações com amount <= 0: 16
Dessas, fraudes: 16


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2736447,212,CASH_OUT,0.0,C1510987794,0.0,0.0,C1696624817,0.00,0.00,1,0
3247298,250,CASH_OUT,0.0,C521393327,0.0,0.0,C480398193,0.00,0.00,1,0
3760289,279,CASH_OUT,0.0,C539112012,0.0,0.0,C1106468520,538547.63,538547.63,1,0
5563714,387,CASH_OUT,0.0,C1294472700,0.0,0.0,C1325541393,7970766.57,7970766.57,1,0
5996408,425,CASH_OUT,0.0,C832555372,0.0,0.0,C1462759334,76759.90,76759.90,1,0


In [12]:
df_validos = limpeza.remover_valores_invalidos(df_sem_nulos)
print(f"Total após remoção de valores inválidos: {len(df_validos)}")
assert (df_validos["amount"] > 0).all()

Total após remoção de valores inválidos: 6362604


In [13]:
# ## 5. Padronização de tipos

In [14]:
print("Tipos antes:")
print(df_validos.dtypes)

df_padronizado = limpeza.padronizar_tipos(df_validos)

print("\nTipos depois:")
print(df_padronizado.dtypes)

Tipos antes:
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Tipos depois:
step                 int64
type              category
amount             float64
nameOrig               str
oldbalanceOrg      float64
newbalanceOrig     float64
nameDest               str
oldbalanceDest     float64
newbalanceDest     float64
isFraud              int64
isFlaggedFraud       int64
dtype: object


In [15]:
# ## 6. Comparação final: bruto vs. limpo


In [16]:
df_limpo = limpeza.limpar_dados(df_bruto)

resumo = pd.DataFrame({
    "etapa": ["Bruto", "Limpo"],
    "total_linhas": [len(df_bruto), len(df_limpo)],
    "total_fraudes": [df_bruto["isFraud"].sum(), df_limpo["isFraud"].sum()],
})
resumo

,etapa,total_linhas,total_fraudes
0,Bruto,6362620,8213
1,Limpo,6362604,8197


In [17]:
# **Conclusão:** a limpeza removeu registros com `amount <= 0`, incluindo
# alguns casos de fraude com valor zero (ver nota em `exploracao.ipynb`).
# Não foram encontradas duplicatas nem nulos nesta base — o dataset PaySim
# já vem consistente nesses aspectos, mas as validações permanecem no
# pipeline por robustez e para lidar com bases futuras que possam não estar
# tão limpas.